# Study Danish v0.6 vs v1.0 — per-donut Zernike comparison (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-05-14
**Last Modified:** 2026-05-14
**Status:** Draft
**Keywords:** AOS, FAM, Danish, donut, version, Zernike, systematics

## Description

The Danish per-donut wavefront estimator was upgraded from v0.6 to
v1.0 mid-run.  Both versions run at `bin_2x` (the summit default).
This notebook compares per-donut Zernike fits between **Danish v0.6**
(chunk0) and **Danish v1.0** (chunks 1, 3, 5 — all bin_2x) on the
**same physical donuts** to look for any version-induced bias.

For each common visit (rotator angle near zero so OCS and CCS
coincide) we match donuts across the two runs by `(detector,
centroid_x_intra, centroid_y_intra)` via a per-CCD KDTree, then
produce:

1. Per-Zernike **density "scatter"** of `Δzk = zk_v1 − zk_v06`  vs
   `zk_v06` (2-D histogram, log color, linear OLS overlay).  Uses
   OCS Zernikes by default.
2. **Matched-donut count map** per `(thx, thy)` bin in both OCS
   and CCS (sanity-check on coverage).
3. Per-Zernike **focal-plane map** of `Δzk` vs `(thx, thy)` in both
   **OCS** (using `zk_OCS` and `thx_OCS / thy_OCS`) and **CCS** (using
   `zk_CCS` and `thx_CCS / thy_CCS`) so any CCD-fixed bias stands
   out separately from any sky-fixed one.

**Output:** Three PDFs + a long-format parquet of matched donut pairs
in `output/study_danish_v0p6_vs_v1/`.


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-14 | Aaron Roodman | Initial version. |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data — visits sidecars & common-visit selection](#data)
5. [Visit-by-visit donut matching](#matching)
6. [Matched-donut count maps](#counts)
7. [Per-Zernike density scatter](#scatter)
8. [Per-Zernike focal-plane maps — OCS](#maps-ocs)
9. [Per-Zernike focal-plane maps — CCS](#maps-ccs)
10. [Outputs — matched-pair parquet & summary table](#output)


<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input donut parquet files -----
# Danish v0.6 single chunk covering 20260315 – 20260409.
donut_parquet_v06 = (
    'output/'
    'aos_fam_danish_triplets_20260315_20260409.parquet')

# Danish v1.0 chunks (bin_2x).  chunk1, chunk3 cover the same date
# range as v0.6; chunk5 sits outside it and will simply produce no
# matches — left in the list so the notebook also runs against the
# full v1.0 dataset.
donut_parquets_v1 = [
    'output/'
    'aos_fam_danish_v1_triplets_bin_2x_20260315_20260327.parquet',
    'output/'
    'aos_fam_danish_v1_triplets_bin_2x_20260331_20260409.parquet',
    'output/'
    'aos_fam_danish_v1_triplets_bin_2x_20260418_20260430.parquet',
]

# ----- Visit filter -----
# Restrict to visits with |rotator_angle| <= rot_max_deg so OCS / CCS
# essentially coincide.
rot_max_deg = 3.0
program_filter = None      # None = no science_program cut

# ----- Donut matching -----
# Two donuts match when their (centroid_x_intra, centroid_y_intra)
# lie within `match_tol_pix` of each other on the same CCD.
match_tol_pix = 5.0

# ----- Plot styling -----
# Density (2-D histogram) "scatter": x = zk_v06, y = Δ = zk_v1 − zk_v06.
scatter_x_plo_pct = 2.0     # x-axis percentile window
scatter_x_phi_pct = 98.0
scatter_y_plo_pct = 1.0     # y-axis percentile window
scatter_y_phi_pct = 99.0
scatter_panels_per_page = 4
scatter_n_bins_per_axis = 100

# Focal-plane median maps (OCS and CCS).
map_n_bins      = 61
map_fp_radius   = 1.8
map_plo_pct     = 2.0
map_phi_pct     = 98.0
map_panels_per_page = 4

# ----- Output -----
output_dir = 'output/study_danish_v0p6_vs_v1'
output_pdf_counts    = f'{output_dir}/study_danish_v0p6_vs_v1_donut_counts.pdf'
output_pdf_scatter   = f'{output_dir}/study_danish_v0p6_vs_v1_scatter.pdf'
output_pdf_maps_ocs  = f'{output_dir}/study_danish_v0p6_vs_v1_focalmaps_OCS.pdf'
output_pdf_maps_ccs  = f'{output_dir}/study_danish_v0p6_vs_v1_focalmaps_CCS.pdf'
output_table_path    = f'{output_dir}/study_danish_v0p6_vs_v1_pairs.parquet'


<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial import cKDTree
from scipy.stats import binned_statistic_2d
from astropy.table import QTable
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.zernike_names import NOLL_NAMES

setup_plotting()

print('Ready.')


<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def _alt_to_deg(alt_arr):
    a = np.asarray(alt_arr, dtype=float)
    if np.nanmax(np.abs(a)) < 2.0 * np.pi + 1e-3:
        return np.rad2deg(a)
    return a


def visits_sidecar_path(donut_parquet_path):
    """Return the matching `*_visits.parquet` for a donut parquet."""
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def select_visits(visits_table, *, rot_max_deg=3.0, program_filter=None):
    """Boolean mask over visits_table for visits to keep."""
    keep = np.ones(len(visits_table), dtype=bool)
    if 'rotator_angle' in visits_table.colnames:
        rot = np.asarray(visits_table['rotator_angle'], dtype=float)
        keep &= np.isfinite(rot) & (np.abs(rot) <= rot_max_deg)
    if program_filter is not None and 'science_program' in visits_table.colnames:
        sp = np.asarray(visits_table['science_program']).astype(str)
        if isinstance(program_filter, (list, tuple, set)):
            prog_set = {str(p) for p in program_filter}
            keep &= np.array([s in prog_set for s in sp])
        else:
            keep &= (sp == str(program_filter))
    return keep


def build_row_group_lookup(parquet_path):
    """`(day_obs, seq_num) -> row_group_idx` from parquet column statistics."""
    pf = pq.ParquetFile(str(parquet_path))
    lookup = {}
    for i in range(pf.num_row_groups):
        meta = pf.metadata.row_group(i)
        d = s = None
        for ci in range(meta.num_columns):
            cmeta = meta.column(ci)
            name = cmeta.path_in_schema
            if name == 'day_obs' and cmeta.statistics is not None:
                d = cmeta.statistics.min
            elif name == 'seq_num' and cmeta.statistics is not None:
                s = cmeta.statistics.min
        if d is not None and s is not None:
            lookup[(int(d), int(s))] = i
    return pf, lookup


def build_v1_visit_lookup(donut_parquets_v1):
    """Pool every v1 chunk's per-visit row-group index into a single dict.

    Returns
    -------
    pf_by_path : dict path -> open ParquetFile
    visit_to_chunk : dict (day_obs, seq_num) -> (path, row_group_idx)
    """
    pf_by_path = {}
    visit_to_chunk = {}
    for p in donut_parquets_v1:
        if not Path(p).exists():
            print(f'(skip missing v1 chunk: {p})')
            continue
        pf, rg = build_row_group_lookup(p)
        pf_by_path[p] = pf
        for key, idx in rg.items():
            if key in visit_to_chunk:
                # Same visit appears in multiple chunks — keep the
                # first one; date-range overlaps should be rare.
                continue
            visit_to_chunk[key] = (p, idx)
    return pf_by_path, visit_to_chunk


def read_visit_donuts(pf, row_group_idx):
    """Load one visit's donuts as a pandas DataFrame from row group idx."""
    return pf.read_row_group(row_group_idx).to_pandas()


def match_donuts_per_ccd(df_a, df_b, *, tol_pix=5.0):
    """Match donuts in df_a and df_b on the same CCD using intra centroids."""
    idx_a_all = []
    idx_b_all = []
    det_a = df_a['detector'].to_numpy(dtype=str)
    det_b = df_b['detector'].to_numpy(dtype=str)
    for det in np.unique(det_a):
        ai = np.where(det_a == det)[0]
        bi = np.where(det_b == det)[0]
        if len(ai) == 0 or len(bi) == 0:
            continue
        xa = df_a['centroid_x_intra'].to_numpy(dtype=float)[ai]
        ya = df_a['centroid_y_intra'].to_numpy(dtype=float)[ai]
        xb = df_b['centroid_x_intra'].to_numpy(dtype=float)[bi]
        yb = df_b['centroid_y_intra'].to_numpy(dtype=float)[bi]
        tree = cKDTree(np.column_stack([xa, ya]))
        dist, k = tree.query(np.column_stack([xb, yb]),
                             distance_upper_bound=tol_pix)
        good = np.isfinite(dist) & (dist < tol_pix)
        idx_a_all.append(ai[k[good]])
        idx_b_all.append(bi[good])
    if not idx_a_all:
        return np.empty(0, dtype=int), np.empty(0, dtype=int)
    return np.concatenate(idx_a_all), np.concatenate(idx_b_all)


def zk_label(j):
    """Pretty label 'Z{j} ({name})' for a pupil Noll index."""
    return f'Z{j} ({NOLL_NAMES.get(j, "?")})'


def stream_pdf_pages(panel_iter, panels_per_page, ncols, page_size,
                     suptitle_fmt, output_pdf, plot_panel):
    """Stream PDF pages with bounded memory."""
    Path(output_pdf).parent.mkdir(parents=True, exist_ok=True)
    panels = list(panel_iter)
    n_total = len(panels)
    n_pages = (n_total + panels_per_page - 1) // panels_per_page
    with PdfPages(output_pdf) as pdf:
        for page in range(n_pages):
            chunk = panels[page * panels_per_page:
                            (page + 1) * panels_per_page]
            nrows = (len(chunk) + ncols - 1) // ncols
            fig, axes = plt.subplots(nrows, ncols, figsize=page_size,
                                     layout='constrained')
            axes = np.atleast_1d(axes).ravel()
            for ax, panel in zip(axes, chunk):
                plot_panel(ax, panel)
            for ax in axes[len(chunk):]:
                ax.set_visible(False)
            fig.suptitle(suptitle_fmt.format(page=page + 1,
                                              n_pages=n_pages),
                         fontsize=12)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f'Wrote {n_pages} pages to {output_pdf}')
    return n_pages


<a id='data'></a>
## 4. Data — visits sidecars & common-visit selection

In [ ]:
# Existence check on the v0.6 parquet + at least one v1 parquet.
if not Path(donut_parquet_v06).exists():
    raise FileNotFoundError(donut_parquet_v06)
if not visits_sidecar_path(donut_parquet_v06).exists():
    raise FileNotFoundError(visits_sidecar_path(donut_parquet_v06))
v1_present = [p for p in donut_parquets_v1 if Path(p).exists()]
if not v1_present:
    raise FileNotFoundError('No v1 donut parquet files found.')
print(f'v0.6 donut parquet: {donut_parquet_v06}')
print(f'v1 donut parquets ({len(v1_present)}):')
for p in v1_present:
    print(f'  {p}')

# Visits sidecars
visits_v06_full = QTable.read(str(visits_sidecar_path(donut_parquet_v06)))
print(f'\nv0.6 visits sidecar: {len(visits_v06_full)} rows')

visits_v1_list = []
for p in v1_present:
    visits_v1_list.append(QTable.read(str(visits_sidecar_path(p))))
visits_v1_full = visits_v1_list[0] if len(visits_v1_list) == 1 else \
    QTable(np.concatenate([v.as_array() for v in visits_v1_list]),
           names=visits_v1_list[0].colnames)
print(f'v1   visits sidecar (concatenated): {len(visits_v1_full)} rows')

# Apply rot / program filter to both sides.
m_v06 = select_visits(visits_v06_full, rot_max_deg=rot_max_deg,
                       program_filter=program_filter)
m_v1  = select_visits(visits_v1_full,  rot_max_deg=rot_max_deg,
                       program_filter=program_filter)
visits_v06 = visits_v06_full[m_v06]
visits_v1  = visits_v1_full[m_v1]
print(f'\nAfter rot/program filter: v0.6 kept {len(visits_v06)}, '
      f'v1 kept {len(visits_v1)}')

keys_v06 = set(zip(np.asarray(visits_v06['day_obs']).astype(int).tolist(),
                   np.asarray(visits_v06['seq_num']).astype(int).tolist()))
keys_v1  = set(zip(np.asarray(visits_v1['day_obs']).astype(int).tolist(),
                   np.asarray(visits_v1['seq_num']).astype(int).tolist()))
common_visits = sorted(keys_v06 & keys_v1)
print(f'\nCommon visits in both runs: {len(common_visits)}')
if not common_visits:
    raise RuntimeError('No common visits found — check input files / cuts.')

# Probe Noll j list.
if 'nollIndices' in visits_v06.colnames:
    iZs = [int(j) for j in
           np.asarray(visits_v06['nollIndices'][0]).tolist()]
else:
    pf_probe = pq.ParquetFile(str(donut_parquet_v06))
    df_probe = pf_probe.read_row_group(0).to_pandas()
    nZk = len(df_probe['zk_OCS'].iloc[0])
    if nZk == 21:
        iZs = list(range(4, 20)) + list(range(22, 27))
    else:
        iZs = list(range(4, 4 + nZk))
print(f'Pupil Noll j (n={len(iZs)}): {iZs}')


<a id='matching'></a>
## 5. Visit-by-visit donut matching

For each common visit we load both versions' donut rows, drop unmatched (no intra↔extra pair), and KDTree-match by `(detector, centroid_x_intra, centroid_y_intra)`.  Both `zk_OCS` and `zk_CCS` plus both coordinate-system field angles are accumulated for the downstream OCS and CCS map sections.

In [ ]:
# Build a per-version row-group lookup so we can stream visit-by-visit.
pf_v06, rg_v06 = build_row_group_lookup(donut_parquet_v06)
pf_by_path, visit_to_chunk = build_v1_visit_lookup(v1_present)

zk_v06_OCS_acc = []
zk_v06_CCS_acc = []
zk_v1_OCS_acc  = []
zk_v1_CCS_acc  = []
thx_OCS_acc    = []
thy_OCS_acc    = []
thx_CCS_acc    = []
thy_CCS_acc    = []
det_acc        = []
dobs_acc       = []
snum_acc       = []
n_matched_per_visit   = []
n_unmatched_per_visit = []

print(f'Streaming {len(common_visits)} common visits...')
for d, s in tqdm(common_visits, desc='matching'):
    if (d, s) not in rg_v06 or (d, s) not in visit_to_chunk:
        continue
    df_v06 = read_visit_donuts(pf_v06, rg_v06[(d, s)])
    v1_path, v1_rg = visit_to_chunk[(d, s)]
    df_v1 = read_visit_donuts(pf_by_path[v1_path], v1_rg)

    df_v06 = df_v06[df_v06['matched_intra_extra'].fillna(False)].reset_index(drop=True)
    df_v1  = df_v1 [df_v1 ['matched_intra_extra'].fillna(False)].reset_index(drop=True)
    if len(df_v06) == 0 or len(df_v1) == 0:
        continue

    # df_v06 is treated as side A; df_v1 as side B.
    i_v06, i_v1 = match_donuts_per_ccd(df_v06, df_v1, tol_pix=match_tol_pix)
    n_matched = len(i_v06)
    n_unmatched = max(len(df_v06), len(df_v1)) - n_matched
    n_matched_per_visit.append(n_matched)
    n_unmatched_per_visit.append(n_unmatched)
    if n_matched == 0:
        continue

    zk_v06_OCS_acc.append(np.stack(df_v06['zk_OCS'].values[i_v06]))
    zk_v06_CCS_acc.append(np.stack(df_v06['zk_CCS'].values[i_v06]))
    zk_v1_OCS_acc.append(np.stack(df_v1 ['zk_OCS'].values[i_v1]))
    zk_v1_CCS_acc.append(np.stack(df_v1 ['zk_CCS'].values[i_v1]))

    # Extra-focal centroid field angles in DEGREES; take them from
    # v0.6 (they should match v1 to within sub-arcsec since the same
    # detection upstream feeds both).
    thx_OCS_acc.append(np.rad2deg(df_v06['thx_OCS_extra'].to_numpy(dtype=float)[i_v06]))
    thy_OCS_acc.append(np.rad2deg(df_v06['thy_OCS_extra'].to_numpy(dtype=float)[i_v06]))
    thx_CCS_acc.append(np.rad2deg(df_v06['thx_CCS_extra'].to_numpy(dtype=float)[i_v06]))
    thy_CCS_acc.append(np.rad2deg(df_v06['thy_CCS_extra'].to_numpy(dtype=float)[i_v06]))

    det_acc.append(df_v06['detector'].to_numpy(dtype=str)[i_v06])
    dobs_acc.append(np.full(n_matched, d, dtype=int))
    snum_acc.append(np.full(n_matched, s, dtype=int))

zk_v06_OCS = np.concatenate(zk_v06_OCS_acc, axis=0) if zk_v06_OCS_acc else np.empty((0, len(iZs)))
zk_v06_CCS = np.concatenate(zk_v06_CCS_acc, axis=0) if zk_v06_CCS_acc else np.empty((0, len(iZs)))
zk_v1_OCS  = np.concatenate(zk_v1_OCS_acc,  axis=0) if zk_v1_OCS_acc  else np.empty((0, len(iZs)))
zk_v1_CCS  = np.concatenate(zk_v1_CCS_acc,  axis=0) if zk_v1_CCS_acc  else np.empty((0, len(iZs)))
thx_OCS = np.concatenate(thx_OCS_acc) if thx_OCS_acc else np.empty(0)
thy_OCS = np.concatenate(thy_OCS_acc) if thy_OCS_acc else np.empty(0)
thx_CCS = np.concatenate(thx_CCS_acc) if thx_CCS_acc else np.empty(0)
thy_CCS = np.concatenate(thy_CCS_acc) if thy_CCS_acc else np.empty(0)
det     = np.concatenate(det_acc)     if det_acc     else np.empty(0, dtype=str)
dobs    = np.concatenate(dobs_acc)    if dobs_acc    else np.empty(0, dtype=int)
snum    = np.concatenate(snum_acc)    if snum_acc    else np.empty(0, dtype=int)

total_matched = int(zk_v06_OCS.shape[0])
total_unmatched = int(sum(n_unmatched_per_visit))
print(f'\nMatched donut pairs: {total_matched:,}')
print(f'Unmatched donuts:    {total_unmatched:,}  '
      f'(≈ {total_unmatched / max(1, total_matched + total_unmatched):.1%})')
if n_matched_per_visit:
    print(f'Per-visit match counts: median = '
          f'{int(np.median(n_matched_per_visit))}, '
          f'range = [{min(n_matched_per_visit)}, '
          f'{max(n_matched_per_visit)}]')


<a id='counts'></a>
## 6. Matched-donut count maps (sanity-check on coverage)

Number of matched donut pairs falling into each `(thx, thy)` bin in
both **OCS** and **CCS** coordinate systems.  Same binning grid as the
per-Zernike map sections so visual comparison is direct.  Coverage
should look like the standard FAM 9-detector fingerprint with
uniformly populated bins inside each chip and gaps between chips.
A log color scale is used because per-bin counts span ~2 orders of
magnitude.


In [ ]:
# Donut-count maps in OCS and CCS, side by side.
from matplotlib.colors import LogNorm

_xb_counts = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)
_yb_counts = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)


def _donut_count_panel(ax, thx_arr, thy_arr, coord_label, vmax_shared):
    counts, _, _ = np.histogram2d(thy_arr, thx_arr,
                                  bins=[_xb_counts, _yb_counts])
    norm = LogNorm(vmin=1, vmax=max(vmax_shared, 2))
    masked = np.ma.masked_where(counts == 0, counts)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white', alpha=0)
    im = ax.imshow(
        masked.T, origin='lower',
        extent=[_xb_counts[0], _xb_counts[-1],
                _yb_counts[0], _yb_counts[-1]],
        cmap=cmap, norm=norm, interpolation='none', aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.75, label='matched donuts / bin')
    ax.set_xlabel(f'thy_{coord_label} [deg]', fontsize=9)
    ax.set_ylabel(f'thx_{coord_label} [deg]', fontsize=9)
    ax.set_title(f'matched donut counts — {coord_label}', fontsize=10)
    ax.tick_params(labelsize=8)
    return float(counts.max())


# Compute counts once on each coord system so we can use a shared
# vmax for the two panels (makes them visually comparable).
_c_ocs, _, _ = np.histogram2d(thy_OCS, thx_OCS, bins=[_xb_counts, _yb_counts])
_c_ccs, _, _ = np.histogram2d(thy_CCS, thx_CCS, bins=[_xb_counts, _yb_counts])
_shared_vmax = float(max(_c_ocs.max(), _c_ccs.max()))

fig_counts, axes = plt.subplots(1, 2, figsize=(13, 6), layout='constrained')
_donut_count_panel(axes[0], thx_OCS, thy_OCS, 'OCS', _shared_vmax)
_donut_count_panel(axes[1], thx_CCS, thy_CCS, 'CCS', _shared_vmax)
fig_counts.suptitle(
    f'Matched donut counts per (thx, thy) bin   '
    f'(n_pairs={total_matched:,},  {map_n_bins}×{map_n_bins} bins, '
    f'|rot|<{rot_max_deg:g}°)',
    fontsize=12)

# Stream the single figure to its own PDF.
Path(output_pdf_counts).parent.mkdir(parents=True, exist_ok=True)
with PdfPages(output_pdf_counts) as _pdf:
    _pdf.savefig(fig_counts, bbox_inches='tight')
print(f'Wrote donut-count map to {output_pdf_counts}')


<a id='scatter'></a>
## 7. Per-Zernike density scatter

Δ = `zk_v1 − zk_v06` (OCS) vs `zk_v06` (OCS), 2-D histogram with a log color scale and a linear OLS overlay clipped to the percentile-bounded plotting box.  Each panel prints slope, intercept, median Δ, σ_MAD, correlation, and pair count for the visible donuts.

In [ ]:
# Per-Zernike density "scatter": Δ = zk_v1 − zk_v06  vs  zk_v06,
# using OCS Zernikes.  Same convention as study_binning: x-axis range
# from a percentile window on the v0.6 values, y-axis from a tighter
# percentile on Δ.
def _scatter_panel(ax, payload):
    j, x, y = payload   # x = zk_v06,  y = zk_v1
    dy = y - x
    mask = np.isfinite(x) & np.isfinite(dy)
    if not np.any(mask):
        ax.set_visible(False)
        return
    x = x[mask]; dy = dy[mask]
    x_lo = float(np.nanpercentile(x,  scatter_x_plo_pct))
    x_hi = float(np.nanpercentile(x,  scatter_x_phi_pct))
    y_lo = float(np.nanpercentile(dy, scatter_y_plo_pct))
    y_hi = float(np.nanpercentile(dy, scatter_y_phi_pct))
    x_pad = 0.02 * max(abs(x_hi - x_lo), 1e-6)
    y_pad = 0.02 * max(abs(y_hi - y_lo), 1e-6)
    x_lo -= x_pad; x_hi += x_pad
    y_lo -= y_pad; y_hi += y_pad

    from matplotlib.colors import LogNorm
    xbins = np.linspace(x_lo, x_hi, scatter_n_bins_per_axis + 1)
    ybins = np.linspace(y_lo, y_hi, scatter_n_bins_per_axis + 1)
    counts, xedges, yedges = np.histogram2d(x, dy, bins=[xbins, ybins])
    norm = LogNorm(vmin=1, vmax=max(counts.max(), 2))
    masked = np.ma.masked_where(counts == 0, counts)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white', alpha=0)
    pcm = ax.pcolormesh(xedges, yedges, masked.T, cmap=cmap, norm=norm,
                        shading='auto')
    plt.colorbar(pcm, ax=ax, shrink=0.85, label='donuts / bin')

    ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.7)
    in_box = (x >= x_lo) & (x <= x_hi) & (dy >= y_lo) & (dy <= y_hi)
    if int(in_box.sum()) >= 5:
        coef = np.polyfit(x[in_box], dy[in_box], 1)
        m, b = float(coef[0]), float(coef[1])
        xf = np.array([x_lo, x_hi])
        ax.plot(xf, m * xf + b, 'r-', lw=1.2, alpha=0.9)
        r = float(np.corrcoef(x[in_box], dy[in_box])[0, 1])
        med = float(np.median(dy[in_box]))
        mad = float(np.median(np.abs(dy[in_box] - med)))
        sigma_mad = 1.4826 * mad
        label = (f'slope={m:+.4f}\n'
                 f'intercept={b:+.4f} μm\n'
                 f'median Δ={med:+.4f} μm\n'
                 f'σ_MAD={sigma_mad:.4f} μm\n'
                 f'r={r:+.4f}\n'
                 f'n={int(in_box.sum())}')
    else:
        label = f'n={int(in_box.sum())}'
    ax.text(0.04, 0.96, label, transform=ax.transAxes, ha='left',
            va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.85,
                      lw=0))
    ax.set_xlim(x_lo, x_hi); ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel('zk_v0.6 (OCS) [μm]', fontsize=9)
    ax.set_ylabel('zk_v1.0 − zk_v0.6 (OCS) [μm]', fontsize=9)
    ax.set_title(zk_label(j), fontsize=10)
    ax.tick_params(labelsize=8)


scatter_panels = [
    (j, zk_v06_OCS[:, idx], zk_v1_OCS[:, idx])
    for idx, j in enumerate(iZs)
]
n_scatter_pages = stream_pdf_pages(
    scatter_panels,
    panels_per_page=scatter_panels_per_page, ncols=2,
    page_size=(11, 9),
    suptitle_fmt=(f'Δzk = v1.0 − v0.6  vs  v0.6   per-donut density (OCS)  '
                  f'(n_pairs={total_matched:,}, '
                  f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
    output_pdf=output_pdf_scatter,
    plot_panel=_scatter_panel)


<a id='maps-ocs'></a>
## 8. Per-Zernike focal-plane maps — OCS

`scipy.stats.binned_statistic_2d` median of Δzk in (thx_OCS, thy_OCS) bins.  Symmetric RdBu_r color scale from the 2-98 % percentile of the binned medians.

In [ ]:
def _make_map_panel(thx_arr, thy_arr, coord_label):
    """Returns a panel-plot function bound to a coord-system's grid."""
    xb = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)
    yb = np.linspace(-map_fp_radius, map_fp_radius, map_n_bins + 1)

    def _panel(ax, payload):
        j, delta = payload
        if len(delta) == 0:
            ax.set_visible(False)
            return
        stat, _, _, _ = binned_statistic_2d(
            thy_arr, thx_arr, delta, statistic='median',
            bins=[xb, yb])
        finite = stat[np.isfinite(stat)]
        if finite.size == 0:
            ax.set_visible(False)
            return
        lo = float(np.nanpercentile(finite, map_plo_pct))
        hi = float(np.nanpercentile(finite, map_phi_pct))
        vlim = max(abs(lo), abs(hi), 1e-6)
        im = ax.imshow(
            stat.T, origin='lower',
            extent=[xb[0], xb[-1], yb[0], yb[-1]],
            cmap='RdBu_r', interpolation='none', aspect='equal',
            vmin=-vlim, vmax=vlim)
        plt.colorbar(im, ax=ax, shrink=0.75, label='Δ [μm]')
        ax.set_xlabel(f'thy_{coord_label} [deg]', fontsize=9)
        ax.set_ylabel(f'thx_{coord_label} [deg]', fontsize=9)
        ax.set_title(zk_label(j) + '  —  v1.0 − v0.6', fontsize=10)
        ax.tick_params(labelsize=8)

    return _panel


# Focal-plane median maps in OCS: zk_OCS Δ vs (thx_OCS, thy_OCS).
panel_ocs = _make_map_panel(thx_OCS, thy_OCS, 'OCS')
map_panels_ocs = [(j, zk_v1_OCS[:, idx] - zk_v06_OCS[:, idx])
                  for idx, j in enumerate(iZs)]
n_map_pages_ocs = stream_pdf_pages(
    map_panels_ocs,
    panels_per_page=map_panels_per_page, ncols=2,
    page_size=(11, 10),
    suptitle_fmt=(f'Focal-plane Δzk = v1.0 − v0.6  (OCS / OCS)  '
                  f'(n_pairs={total_matched:,}, '
                  f'{map_n_bins}×{map_n_bins} bins, '
                  f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
    output_pdf=output_pdf_maps_ocs,
    plot_panel=panel_ocs)


<a id='maps-ccs'></a>
## 9. Per-Zernike focal-plane maps — CCS

Same as §7 but in CCS: uses `zk_CCS` for the Δ values *and* (thx_CCS, thy_CCS) for the bin centers, so any CCD-fixed (rather than sky-fixed) bias is what shows up here.

In [ ]:
# Focal-plane median maps in CCS: zk_CCS Δ vs (thx_CCS, thy_CCS).
panel_ccs = _make_map_panel(thx_CCS, thy_CCS, 'CCS')
map_panels_ccs = [(j, zk_v1_CCS[:, idx] - zk_v06_CCS[:, idx])
                  for idx, j in enumerate(iZs)]
n_map_pages_ccs = stream_pdf_pages(
    map_panels_ccs,
    panels_per_page=map_panels_per_page, ncols=2,
    page_size=(11, 10),
    suptitle_fmt=(f'Focal-plane Δzk = v1.0 − v0.6  (CCS / CCS)  '
                  f'(n_pairs={total_matched:,}, '
                  f'{map_n_bins}×{map_n_bins} bins, '
                  f'|rot|<{rot_max_deg:g}°)  [{{page}}/{{n_pages}}]'),
    output_pdf=output_pdf_maps_ccs,
    plot_panel=panel_ccs)


<a id='output'></a>
## 10. Outputs

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
out = {
    'day_obs':     dobs,
    'seq_num':     snum,
    'detector':    det,
    'thx_OCS_deg': thx_OCS,
    'thy_OCS_deg': thy_OCS,
    'thx_CCS_deg': thx_CCS,
    'thy_CCS_deg': thy_CCS,
}
for idx, j in enumerate(iZs):
    out[f'zk_v06_OCS_z{j}']  = zk_v06_OCS[:, idx]
    out[f'zk_v1_OCS_z{j}']   = zk_v1_OCS[:, idx]
    out[f'delta_OCS_z{j}']   = zk_v1_OCS[:, idx]  - zk_v06_OCS[:, idx]
    out[f'zk_v06_CCS_z{j}']  = zk_v06_CCS[:, idx]
    out[f'zk_v1_CCS_z{j}']   = zk_v1_CCS[:, idx]
    out[f'delta_CCS_z{j}']   = zk_v1_CCS[:, idx]  - zk_v06_CCS[:, idx]
df_out = pd.DataFrame(out)
df_out.to_parquet(output_table_path)
print(f'Saved {len(df_out):,} matched donut pairs -> {output_table_path}')
print(f'Columns: {len(df_out.columns)}')

# Per-Zernike summary table of v1 − v0.6 deltas (OCS).
summary_rows = []
for idx, j in enumerate(iZs):
    delta = zk_v1_OCS[:, idx] - zk_v06_OCS[:, idx]
    delta = delta[np.isfinite(delta)]
    if len(delta) == 0:
        continue
    med = float(np.median(delta))
    mad = float(np.median(np.abs(delta - med)))
    summary_rows.append({
        'j': int(j),
        'name': NOLL_NAMES.get(j, '?'),
        'n': int(len(delta)),
        'mean':       float(np.mean(delta)),
        'median':     med,
        'sigma_mad':  1.4826 * mad,
    })
df_summary = pd.DataFrame(summary_rows)
with pd.option_context('display.max_rows', None,
                       'display.float_format', '{:+.4f}'.format):
    display(df_summary)
